In [17]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import optuna


from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

sys.path.append(str(Path().resolve().parent))
import src.data_processing.data_processing as dp
import src.data_processing.building_dataset as bd

/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = dp.load_data('../data/raw/grades.csv')

df = dp.clean_data(df)
df = dp.process_data(df)

df_final = bd.make_features_for_status(df)
df_final.head()



/mnt/d/ml_project/ml_project_2026/src/data_processing/data_processing.py:23: DtypeWarning: Columns (0: course, 1: module, 2: academic_year) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep = ';')


,student__id_hash,avg_grade,count_grades,proportion_retake,proportion_retake_com,student_status,course_1,course_2,course_3,course_4,...,place_type_Внеконкурсное поступление,place_type_Коммерческие,place_type_Коммерческие за счет средств вуза,place_type_Коммерческие за счет средств вуза для иностранных граждан,place_type_Коммерческие места для иностранных граждан,place_type_Места с оплатой стоимости обучения на договорной основе,"place_type_Места, обеспеченные государственным финансированием",place_type_По межправительственным соглашениям,place_type_Сетевой студент,place_type_Целевые
0,00000d86dad0d3c197341f776bb61a1d,8.166667,24,0.000000,0.000000,graduated,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0001dd443aff5c0800694730323c3d7d,6.973913,115,0.034783,0.008696,study,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,00035b28bedc6d3a0336cb61a841d79b,7.214286,14,0.000000,0.000000,study,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,00044a704df898b9a87faf40795d62b6,8.454545,22,0.000000,0.000000,graduated,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0004d03d5c41e468a521ed8a964fe889,7.250000,16,0.062500,0.000000,study,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df_final = dp.str_to_int_student_status(df_final)

In [ ]:
X = df_final.drop(columns=['student_id_hash', 'student_status'])
y = df_final['student_status']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [19]:

def objective(trial, X, y):
    params = {

        'max_depth': trial.suggest_int('max_depth', 2, 30),

        'min_samples_split': trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),

        'min_samples_leaf': trial.suggest_int(
            'min_samples_leaf',
            1,
            20
        ),

        'criterion': trial.suggest_categorical(
            'criterion',
            ['gini', 'entropy', 'log_loss']
        ),

        'max_features': trial.suggest_categorical(
            'max_features',
            [None, 'sqrt', 'log2']
        )
    }

    model = DecisionTreeClassifier(
        **params,
        random_state=42
    )

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1'
    ).mean()

    return score


def tune_decision_tree(
    X,
    y,
    n_trials: int = 50
):

   
    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective(trial, X, y),
        n_trials=n_trials
    )

    best_params = study.best_params

    print("Лучшие параметры")
    print(best_params)

    print()
    print(f"Лучший score: {study.best_value:.4f}")

    best_model = DecisionTreeClassifier(
        **best_params,
        random_state=42
    )

    best_model.fit(X, y)

    return best_model, study

best_model, study = tune_decision_tree(
    X_train,
    y_train,
    n_trials=100
)

[I 2026-05-18 15:27:48,660] A new study created in memory with name: no-name-2ea35b6f-8251-41d0-9656-1fac8219821f
[I 2026-05-18 15:27:49,639] Trial 0 finished with value: 0.011470936416938414 and parameters: {'max_depth': 8, 'min_samples_split': 9, 'min_samples_leaf': 9, 'criterion': 'entropy', 'max_features': 'log2'}. Best is trial 0 with value: 0.011470936416938414.
[I 2026-05-18 15:27:52,380] Trial 1 finished with value: 0.8047647649632422 and parameters: {'max_depth': 25, 'min_samples_split': 15, 'min_samples_leaf': 17, 'criterion': 'log_loss', 'max_features': None}. Best is trial 1 with value: 0.8047647649632422.
[I 2026-05-18 15:27:52,744] Trial 2 finished with value: 0.0 and parameters: {'max_depth': 4, 'min_samples_split': 19, 'min_samples_leaf': 16, 'criterion': 'log_loss', 'max_features': 'log2'}. Best is trial 1 with value: 0.8047647649632422.
[I 2026-05-18 15:27:55,473] Trial 3 finished with value: 0.8039210413054152 and parameters: {'max_depth': 13, 'min_samples_split': 4,

Лучшие параметры
{'max_depth': 6, 'min_samples_split': 14, 'min_samples_leaf': 13, 'criterion': 'log_loss', 'max_features': None}

Лучший score: 0.8153


In [20]:
best_model_DecisionTree, model_DecisionTree = best_model, study

In [ ]:



def objective_rf(trial, X, y):


    params = {
        'n_estimators': trial.suggest_int(
            'n_estimators',
            50,
            500
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            2,
            30
        ),

        'min_samples_split': trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),

        'min_samples_leaf': trial.suggest_int(
            'min_samples_leaf',
            1,
            20
        ),

        'max_features': trial.suggest_categorical(
            'max_features',
            ['sqrt', 'log2', None]
        ),

        'bootstrap': trial.suggest_categorical(
            'bootstrap',
            [True, False]
        )
    }

    model = RandomForestClassifier(
        **params,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    return score


def tune_random_forest(
    X,
    y,
    n_trials: int = 50
):

    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective_rf(trial, X, y),
        n_trials=n_trials
    )

    best_params = study.best_params


    best_model = RandomForestClassifier(
        **best_params,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    best_model.fit(X, y)

    return best_model, study

best_rf, study_rf = tune_random_forest(
    X_train,
    y_train,
    n_trials=100
)

[I 2026-05-18 15:38:14,983] A new study created in memory with name: no-name-0c808adf-beab-42d8-a41e-16da45044a91
[I 2026-05-18 15:39:29,072] Trial 0 finished with value: 0.7197282979493695 and parameters: {'n_estimators': 120, 'max_depth': 15, 'min_samples_split': 12, 'min_samples_leaf': 6, 'max_features': None, 'bootstrap': True}. Best is trial 0 with value: 0.7197282979493695.
[I 2026-05-18 15:42:37,772] Trial 1 finished with value: 0.6014915143337534 and parameters: {'n_estimators': 378, 'max_depth': 10, 'min_samples_split': 16, 'min_samples_leaf': 1, 'max_features': None, 'bootstrap': False}. Best is trial 0 with value: 0.7197282979493695.
[W 2026-05-18 15:42:58,547] Trial 2 failed with parameters: {'n_estimators': 470, 'max_depth': 23, 'min_samples_split': 17, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'bootstrap': False} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-

KeyboardInterrupt: 

In [22]:

def objective_gb(trial, X, y):


    params = {

        'n_estimators': trial.suggest_int(
            'n_estimators',
            50,
            500
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.001,
            0.3,
            log=True
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            2,
            10
        ),

        'min_samples_split': trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),

        'min_samples_leaf': trial.suggest_int(
            'min_samples_leaf',
            1,
            20
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.5,
            1.0
        ),

        'max_features': trial.suggest_categorical(
            'max_features',
            ['sqrt', 'log2', None]
        )
    }

    model = GradientBoostingClassifier(
        **params,
        random_state=42
    )

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    return score


def tune_gradient_boosting(
    X,
    y,
    n_trials: int = 50
):


    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective_gb(trial, X, y),
        n_trials=n_trials
    )

    best_params = study.best_params


    best_model = GradientBoostingClassifier(
        **best_params,
        random_state=42
    )

    best_model.fit(X, y)

    return best_model, study
best_gb, study_gb = tune_gradient_boosting(
    X_train,
    y_train,
    n_trials=100
)

[I 2026-05-18 15:47:10,899] A new study created in memory with name: no-name-18578607-7189-4785-bdb6-cfe9b4c147b9
[W 2026-05-18 15:47:25,045] Trial 0 failed with parameters: {'n_estimators': 218, 'learning_rate': 0.013348861126938581, 'max_depth': 2, 'min_samples_split': 15, 'min_samples_leaf': 1, 'subsample': 0.874322879291535, 'max_features': None} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_23971/4018207866.py", line 78, in <lambda>
    lambda trial: objective_gb(trial, X, y),
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23971/4018207866.py", line 54, in objective_gb
    score = cross_val_score(
            ^^^^^^^^^^^^^^^^
  File "/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages/sk

KeyboardInterrupt: 

In [ ]:
from sklearn.preprocessing import StandardScaler
def objective_svm(trial, X, y):

    X_scaled = StandardScaler().fit_transform(X)
    params = {

        'C': trial.suggest_float(
            'C',
            1e-3,
            100,
            log=True
        ),

        'kernel': trial.suggest_categorical(
            'kernel',
            ['linear', 'rbf', 'poly']
        ),

        'gamma': trial.suggest_categorical(
            'gamma',
            ['scale', 'auto']
        ),

        'degree': trial.suggest_int(
            'degree',
            2,
            5
        )
    }

    model = SVC(
        **params,
        class_weight='balanced',
        probability=True,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_scaled,
        y,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    return score


def tune_svm(
    X,
    y,
    n_trials: int = 30
):
    X_scaled = StandardScaler().fit_transform(X)

    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective_svm(trial, X_scaled, y),
        n_trials=n_trials
    )

    best_params = study.best_params

  

    best_model = SVC(
        **best_params,
        class_weight='balanced',
        probability=True,
        random_state=42
    )

    best_model.fit(X_scaled, y)

    return best_model, study

best_svm, study_svm = tune_svm(
    X_train,
    y_train,
    n_trials=50
)

In [ ]:


def objective_knn(trial, X, y):


    params = {

        'n_neighbors': trial.suggest_int(
            'n_neighbors',
            1,
            50
        ),

        'weights': trial.suggest_categorical(
            'weights',
            ['uniform', 'distance']
        ),

        'metric': trial.suggest_categorical(
            'metric',
            ['minkowski', 'euclidean', 'manhattan']
        ),

        'p': trial.suggest_int(
            'p',
            1,
            3
        )
    }

    model = KNeighborsClassifier(**params)

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    return score


def tune_knn(
    X,
    y,
    n_trials: int = 50
):


    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective_knn(trial, X, y),
        n_trials=n_trials
    )

    best_params = study.best_params


    best_model = KNeighborsClassifier(**best_params)

    best_model.fit(X, y)

    return best_model, study

best_knn, study_knn = tune_knn(
    X_train,
    y_train,
    n_trials=50
)

In [ ]:

models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': best_model_DecisionTree,
    'Random Forest': best_rf,
    'Gradient Boosting': best_gb ,
    'SVM': best_svm,
    'KNN': best_knn
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    

    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

best_model = results_df.loc[results_df['F1-Score'].idxmax(), 'Model']
print(f"\nЛучшая модель по F1-Score: {best_model}")